In [ ]:
import pandas as pd

df = pd.read_csv("/home/hvgupta/COMP4222/COMP4222_F25-26/src/one_year_company_info.csv")

In [ ]:
df.isnull().sum()

In [ ]:
len(df["Symbol"].unique())

In [ ]:
df.columns

In [ ]:
df["Weight"]

The current data in this file is outdated, I have mostly gotten it since it has the ticker for different stocks.

Some of the columns here are interesting and can be used to in our analysis

I think we can focus on using the following metrics for edges/nodes?(other than the tickers or the identifiers):
  - Market Cap: could show the much volatility there could be 
  - Weight: %maket cap of the company to the total market cap of S&P500


(for now I am gathering data for 500 companies, but we can choose a subset if we want)

we would definetly need to add more

In [ ]:
import pandas as pd
import yfinance as yf
import ta



In [ ]:
aapl = yf.download("AAPL", start="2023-01-01", end="2025-11-01")
msft = yf.download("MSFT", start="2023-01-01", end="2025-11-01")

In [ ]:
h = pd.merge(aapl, msft, left_index=True, right_index=True, suffixes=("_AAPL", "_MSFT"))

In [ ]:
h

In [ ]:
h.corr()

In [ ]:
l = h.copy()

In [ ]:
# Add a new column to the entire dataframe for the filtered date range
l.loc['2023-01-01':'2023-03-31', ('new_column', 'AAPL')] = 'your_value'

In [ ]:
l

In [ ]:
h.loc[:,"Close"]

In [ ]:
with open("test.csv", "w") as f:
    f.write(h.to_csv())

In [ ]:
# Example: download 2 years of data for one stock
data = yf.download("AAPL", start="2023-01-01", end="2025-01-01")

# --- Derived features ---
data["return_1d"] = data["Close"].pct_change()
data["return_5d"] = data["Close"].pct_change(5)
data["volatility_20d"] = data["Close"].pct_change().rolling(20).std()

# Trend features
data["ma_5"] = data["Close"].rolling(5).mean()
data["ma_20"] = data["Close"].rolling(20).mean()
data["ma_ratio"] = data["ma_5"] / data["ma_20"]

# RSI and MACD (from ta library)
data["rsi"] = ta.momentum.RSIIndicator(data["Close"], window=14).rsi()
macd = ta.trend.MACD(data["Close"])
data["macd"] = macd.macd()
data["macd_signal"] = macd.macd_signal()

# Volume-related
data["obv"] = ta.volume.OnBalanceVolumeIndicator(data["Close"], data["Volume"]).on_balance_volume()
data["vol_zscore"] = (data["Volume"] - data["Volume"].rolling(20).mean()) / data["Volume"].rolling(20).std()

# Price range ratio
data["price_range"] = (data["High"] - data["Low"]) / data["Close"]

In [ ]:
data

In [ ]:
import pandas as pd

df = pd.read_csv("./src/one_year_company_info.csv")

In [ ]:
AAPL_EODs = df[df["Symbol"] == "AAPL"]

In [ ]:
AAPL_EODs

In [ ]:
import talib

ROCP_5 = talib.ROCP(AAPL_EODs["Close"].to_numpy(), timeperiod=5)
ROCP_20 = talib.ROCP(AAPL_EODs["Close"].to_numpy(), timeperiod=20)
NATR_5 = talib.NATR(AAPL_EODs["High"].to_numpy(), AAPL_EODs["Low"].to_numpy(), AAPL_EODs["Close"].to_numpy(), timeperiod=5)

In [ ]:
NATR_5

In [ ]:
talib.MOM(AAPL_EODs["Close"].to_numpy(), timeperiod=5)
talib.MOM

In [ ]:
talib.ADXR(AAPL_EODs["High"].to_numpy(), AAPL_EODs["Low"].to_numpy(), AAPL_EODs["Close"].to_numpy(), timeperiod=5)

In [ ]:
talib.BETA()

In [ ]:
import pandas as pd
import requests
from io import StringIO

# Add headers to avoid 403 Forbidden error
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
response = requests.get(url, headers=headers)
sp500 = pd.read_html(StringIO(response.text))[0]
tickers = sp500['Symbol'].tolist()


In [ ]:
sp500

In [ ]:
crta_stock_data = yf.download("CTRA", start="2020-01-01", end="2025-11-01")

In [ ]:
sp500[sp500["Founded"].str.match(r"\d{4} \(.+\)")]

In [ ]:
ctra = yf.Ticker("CTRA")


In [ ]:
q_income = ctra.quarterly_income_stmt
q_balance = ctra.quarterly_balance_sheet
q_earnings = ctra.quarterly_balance_sheet

In [ ]:
prices = crta_stock_data.loc["2020-01-01":"2021-01-01", "Close"]

In [ ]:
ctra.get_history_metadata()

In [ ]:
import yfinance as yf

ticker = yf.Ticker("CTRA")
q_earnings = ticker.get_earnings_dates()
q_earnings.sort_index()


In [ ]:
ticker.

In [3]:
import requests
url = "https://www.sec.gov/files/company_tickers.json"
headers = {'User-Agent': 'YourAppName/1.0 (hvgupta@outlook.in)'}  # Replace with your details
response = requests.get(url, headers=headers)
if response.status_code != 200:
    raise Exception(f"Failed to fetch data: {response.status_code}")


In [4]:
ticker_to_cik_map = {info['ticker']: str(info['cik_str']).zfill(10) for info in response.json().values()}


In [5]:
import requests
def get_sec_facts(cik):
    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"
    headers = {'User-Agent': 'YourAppName/1.0 (your.email@example.com)'}  # Replace with your details
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        return response.json()
    else:
        raise Exception(f"Failed to fetch data: {response.status_code}")

In [6]:
facts = get_sec_facts(ticker_to_cik_map["CTRA"])

In [7]:
import pandas as pd

def extract_quarterly_data(facts: dict, metric_name: str, unit: str) -> pd.DataFrame:
       
    if "us-gaap" not in facts:
        return pd.DataFrame()
    
    if metric_name not in facts["us-gaap"]:
        return pd.DataFrame()
    
    if unit not in facts["us-gaap"][metric_name]["units"]:
        return pd.DataFrame()
    
    data = facts["us-gaap"][metric_name]["units"][unit]
    df = pd.DataFrame(data)
    df['end'] = pd.to_datetime(df['end'])
    df = df.sort_values(by='end').reset_index(drop=True)
    
    return df

In [8]:
x = extract_quarterly_data(facts["facts"], 'EarningsPerShareBasic', 'USD/shares')

In [9]:
x["end"] = pd.to_datetime(x["end"])
x["start"] = pd.to_datetime(x["start"])


In [ ]:
equity_table = extract_quarterly_data(facts["facts"], 'StockholdersEquity', 'USD')
shares_table = extract_quarterly_data(facts["facts"], 'CommonStockSharesIssued', 'shares')

In [ ]:
bv_df = pd.merge(equity_table, shares_table, on=["end","fy", "fp"], suffixes=("_equity", "_shares"))

In [ ]:
bv_df

In [ ]:
from src.gather_company_info import get_PE_ratio_data
from src.market_data_fetcher import *

In [ ]:
PE_info = get_PE_ratio_data("CTRA", get_ticker_historical_prices("CTRA", "2020-01-01", "2025-01-01"), facts["facts"])

In [10]:
eps_table = extract_quarterly_data(facts["facts"], 'EarningsPerShareBasic', 'USD/shares')

In [11]:
eps_table["start"] = pd.to_datetime(eps_table["start"], errors="coerce")
eps_table["end"] = pd.to_datetime(eps_table["end"], errors="coerce")

In [12]:
eps_table = eps_table.dropna(subset=["start", "end", "val"])

In [14]:
filtered_eps = pd.DataFrame(columns=["start","end", "eps", "fp"])

In [15]:
from pandas import Timestamp
def determine_quarter(end_date: Timestamp):
    return end_date.quarter

def get_start_and_end_of_quarter(year: int, quarter: int):
    if quarter == 1:
        return Timestamp(year=year, month=1, day=1), Timestamp(year=year, month=3, day=31)
    elif quarter == 2:
        return Timestamp(year=year, month=4, day=1), Timestamp(year=year, month=6, day=30)
    elif quarter == 3:
        return Timestamp(year=year, month=7, day=1), Timestamp(year=year, month=9, day=30)
    elif quarter == 4:
        return Timestamp(year=year, month=10, day=1), Timestamp(year=year, month=12, day=31)
    else:
        raise ValueError("Quarter must be between 1 and 4")

In [ ]:
y_q_to_eps_map = {}
for i, row in eps_table.iterrows():
    row_dict = {
        "start": row["start"],
        "end": row["end"],
        "eps": row["val"],
        "fp": row["fp"],
    }
    check_quarter = determine_quarter(row["end"])
    if row["fp"] == "FY":
        
        if check_quarter == 4:
            ammended_period = row["frame"]
            if pd.isna(ammended_period) or (len(ammended_period) == 6): # only CY{YYYY} -> this just means that it is the full year eps
                new_row_dict = row_dict.copy()
                new_row_dict["fp"] = "Q4"
                new_row_dict["eps"] = row["val"] - y_q_to_eps_map.get((row["end"].year, 1), 0) - y_q_to_eps_map.get((row["end"].year, 2), 0) - y_q_to_eps_map.get((row["end"].year, 3), 0)
                y_q_to_eps_map[(new_row_dict["end"].year, "Q4")] = new_row_dict
            else:

                y = row["frame"][2:6]
                q = int(row["frame"][-1])
                if (int(y), f"Q{q}") in y_q_to_eps_map:
                    y_q_to_eps_map[(int(y), f"Q{q}")]["eps"] = row_dict["eps"]
                else:
                    start, end = get_start_and_end_of_quarter(int(y), q)
                    y_q_to_eps_map[(int(y), f"Q{q}")] = {
                        "start": start,
                        "end": end,
                        "eps": row_dict["eps"],
                        "fp": f"Q{q}"
                    }
                    
                continue
        else:
            row_dict["fp"] = f"Q{check_quarter}" # some error which is not supposed to happen, just fall back to this (could be a source of error)
    
    
    y_q_to_eps_map[(row_dict["end"].year, row_dict["fp"])] = row_dict

filtered_eps = pd.concat([filtered_eps, pd.DataFrame(list(y_q_to_eps_map.values()))], ignore_index=True)

here


TypeError: 'float' object is not subscriptable

In [ ]:
filtered_eps[filtered_eps["fp"] == "Q4"]